# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [108]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [109]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [110]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [111]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [112]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [113]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [114]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [115]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, one of the most common issues with student loans appears to be problems related to dealing with lenders or servicers. Specific sub-issues include receiving bad information about loans, incorrect or inaccurate reporting of loan status or balances, trouble with how payments are handled (such as being unable to apply extra payments to the principal), and the mismanagement or mishandling of loans—including improper transfers, unnotified changes, and errors in loan account information. \n\nIn summary, a frequently reported and significant issue is the mishandling and miscommunication by loan servicers, leading to errors in balances, interest, or account status, which deeply affects borrowers' financial stability and credit reports."

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, some complaints did not get handled in a timely manner. Specifically, at least two complaints indicate they were not addressed promptly:\n\n- One complaint received on 03/28/25 by MOHELA was marked as "Not timely," meaning it was handled late.\n- Another complaint received on 04/24/25 by Maximus Federal Services, Inc. was resolved with the response "Closed with explanation," but the content suggests ongoing issues and delays.\n\nOn the other hand, several complaints received prompt responses marked as "Yes" for timely response, but the existence of late and unresolved complaints indicates that not all complaints were handled in a timely manner.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a combination of factors including insufficient information about repayment terms, unexpected or unauthorized loan transfers, lack of clear communication from loan servicers, difficulties in managing repayment plans (such as interest accumulation and inability to pay higher amounts), and administrative issues like errors or mismanagement by loan providers. Additionally, some borrowers faced hardships from economic circumstances, stagnant wages, or financial mismanagement, which made monthly payments unmanageable. Issues with how payments are applied, limited options to accelerate payoff, and perceived predatory practices by servicers also contributed to their inability to fulfill repayment obligations.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [116]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [117]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the lender or servicer, specifically problems related to miscommunication, incorrect information, or alleged unfair practices. The issues cited include disagreements over fees, difficulties in applying payments correctly (such as being unable to pay off in smaller amounts or redirect payments to principal), and receiving bad or incomplete information about loan balances or terms.\n\nIn summary, the most common issue involves challenges in communication and the handling of loan payments or information by lenders or servicers.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that several complaints were handled in a timely manner, as indicated by the response status "Yes" under "Timely response?" for each complaint. Specifically:\n\n- The complaint with ID 13197090 was responded to promptly with a "Closed with explanation" response within the expected timeframe.\n- The complaint with ID 12792958 also received a response marked as "Closed with explanation" in a timely manner.\n- Similarly, the complaint with ID 13160766 was addressed within the appropriate timeframe.\n- The complaint with ID 13410623 was also responded to in a timely manner.\n\nThere is no evidence in the provided data of any complaints not being handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People fail to pay back their loans for several reasons highlighted in the complaints:\n\n1. **Problems with payment plans or forbearances:** Some borrowers experience issues with their payment plans, such as being steered into the wrong types of forbearances or having their forbearance requests ignored by the servicers.\n\n2. **Poor communication from loan servicers:** There are cases where borrowers are not properly informed about changes to their loans, such as transfers to new companies (e.g., Aidvantage), or the discontinuation of autopay, which can lead to missed payments and negative credit impacts.\n\n3. **Errors and technical issues:** Frequent problems like reversed payments, incorrect account information, or failure of the servicer to process payments correctly contribute to missed payments.\n\n4. **Lack of response or support from the servicer:** When borrowers request assistance or apply for deferments or forbearances, delays or lack of responses from the servicers result

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer:

BM25 excels in instances where exact keyword matching (no synonyms) is needed. Specific query topics that might use BM25 are health/medicine, law, and/or math.

An example query could be: "What medicine should I take if I have a fever?"

To answer this query, exact keyword match is critical and documents titled like "Fever remedies" or "How to cure a fever" will most likely be retrieved. 

Normal embedding-based retrieval might return medicine for other sicknesses, or natural remedies for fevers (sleep, rest, etc), instead of _specific medications._

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [118]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [119]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, a common issue with loans, particularly student loans, involves problems with loan servicing, such as errors in loan balances, misapplied payments, incorrect or incomplete information, and inadequate communication from lenders or servicers. These issues often lead to disputes, incorrect credit reporting, and legal or privacy concerns.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, there are indications that some complaints experienced delays or unresolved issues. For example:\n\n- The complaint regarding loan account review and potential violations of FERPA has been open for nearly 18 months with no resolution, which suggests it was not handled in a timely manner.\n- The complaint about payments not being applied to the loan account (with a previous reference) indicates ongoing issues, though it\'s not explicitly stated if it was timely handled or not.\n\nHowever, the records explicitly mention that responses to two complaints about federal student loan servicing were marked as "Timely response? Yes." Despite that, the existence of unresolved or delayed complaints (like the nearly 18-month open issue) indicates that some complaints did not get handled promptly.\n\nIn summary, yes: **some complaints did not get handled in a timely manner**, especially the one that was pending for nearly 18 months without resolution.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a combination of factors such as lack of clear information about repayment obligations, unexpected transfers of loan control, poor communication from lenders or servicers, and the accumulation of interest that makes repaying the principal amount increasingly difficult. Additionally, some borrowers are unaware of the terms and conditions of their loans, including how interest accrues, and may face financial hardships that prevent them from making timely payments. In some cases, borrowers are misled or not properly informed about their repayment options or the transfer of their loans, which further complicates their ability to repay.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [120]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [121]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and data, the most common issues with loans tend to revolve around mismanagement, misreporting, and problems with loan servicers. Specifically, frequent issues include:\n\n- Errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n- Errors and inaccuracies in credit reporting, including incorrect delinquency status and account reporting.\n- Poor communication, lack of notices about loan transfers or changes, and inadequate customer service.\n- Problems with loan repayment plans, such as being steered into forbearance or deferment that leads to interest accumulation and increased balances.\n- Mishandling of applications for loan forgiveness or income-driven repayment plans.\n- Unauthorized or unverified collection efforts and illegal credit reporting.\n- Lack of transparency and inadequate information about account status or interest calculations.\n\nOverall, the most common issue appears to be **mismanagement and errors rel

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints and responses, yes, some complaints were not handled in a timely manner. Several complaints explicitly mention delays, lack of response, or failure to respond within expected timeframes. For example:\n\n- Complaint ID: 12709087 (MOHELA, CA): The complaint states "it is currently over 2-3 weeks and I am still having this issue."\n- Complaint ID: 12654977 (MOHELA, MD): The response was marked "No" with regard to timely response.\n- Complaint ID: 12739706 (Mohela, KY): The response was also marked "No" for timely response.\n- Complaint ID: 13062402 (Nelnet, MI): Despite being marked "Yes" for timely response, the complaint indicates ongoing issues with no resolution after more than a month.\n- Multiple other complaints specify responses taking more than the expected time, or no response at all.\n\nIn conclusion, there have been complaints that did not get handled in a timely manner, indicating delays and unresolved issues.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to systemic issues and misconduct by loan servicers and lenders, as indicated in the provided complaints. These issues include:\n\n- Lack of proper communication from servicers regarding overdue status, default notices, or delinquency warnings, often leading borrowers to be unaware of their repayment obligations until they face severe consequences like credit reporting or collections.\n- Errors and mismanagement in loan servicing, such as incorrect account status reports, misapplied payments, or failing to follow regulations for diligent collection efforts, which can cause credit scores to drop unexpectedly.\n- Misleading or coercive practices, such as steering borrowers into long-term forbearances instead of informing them about income-driven repayment plans, rehabilitation programs, or other legal options that could reduce or forgive debt.\n- Systemic failures including improper reporting of default or delinquency, failure to provi

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:

Multi-Query retrieval can be beneficial by retrieving documents that are still on topic with the original query, but expand the scope and give more context/information through answering the newly generated queries. The reformulated queries might capture different phrasings or interpretations of the original query that could be beneficial.

For example, using the fever scenario again, the original query could be: 
    "What do I do if I have a fever?"

Reformulated queries might include: 
    "Why do we get fevers?"
    "Step by step instructions to curing a fever?"
    "How to prevent fevers in the future"
    "Over-the-counter medicines to lower a fever"
    "When to go to the doctor when having a fever"

Each of these queries might return different documents than the original query would, but they are all still relevant to the original query.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [122]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [123]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [124]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [125]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [126]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be related to errors and misconduct in federal student loan servicing. Specific recurring problems include incorrect information on credit reports, misapplication of payments, wrongful denials of payment plans, discrepancies in loan balances and interest rates, and issues with collection and verification of debts.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that several complaints were not handled in a timely manner. Specifically, the complaints related to the student loan issues with MOHELA (Complaint IDs 12709087 and 12935889) indicate that the responses were "No" in the "Timely response?" field, meaning they were not handled promptly. Additionally, the complaint about the dispute settlement with Nelnet (Complaint ID 13205525) was responded to within the expected timeframe ("Yes" in "Timely response?"). \n\nTherefore, yes, some complaints—particularly those regarding MOHELA—did not get handled in a timely manner.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n1. Lack of proper communication or notification from loan servicers about payment obligations, as indicated by complaints about not being notified when payments were due or about changes in loan ownership.\n2. Financial hardship or severe economic difficulties that made it impossible to make timely payments, such as unemployment or inability to find employment in their field.\n3. Misrepresentation or lack of transparency from educational institutions and loan providers regarding the long-term financial consequences, job prospects after graduation, and the sustainability of the school’s operations.\n4. Relying on deferment and forbearance options that increased interest and debt over time.\n5. Disputes over the legitimacy or ownership of the debt, including issues related to the legal verification of loans and deceptive practices by collection agencies.\n6. Personal health issues or other personal circumstances th

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [127]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [128]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [44]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided data, appears to be dealing with the loan servicer or lender, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and problems with how payments are being handled. Several complaints highlight issues such as receiving bad information about loans, inability to properly apply payments to principal, inaccurate reporting of delinquency, and mishandling of loan transfers or consolidations. \n\nIn summary, a predominant and recurring problem is the mismanagement and poor communication from loan servicers, which leads to misapplied payments, incorrect account information, and difficulties in resolving repayment issues.'

In [45]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, there are several instances indicating complaints not handled in a timely manner. For example:\n\n- One complaint (#12935889) about Mohela was marked as "Timely response?": No.\n- Another (#12744910) regarding inaccuracies in reporting and an ongoing dispute was "Timely response?": Yes, but the complaint was about inaccurate reporting and delays in correction, suggesting the issue persisted over time.\n- Multiple complaints (#12739706, #13062402, #13126709, #13127090, and others) mention delays, extended wait times, or responses that were not addressed promptly, with some even explicitly stating they did not receive responses within expected timeframes.\n- There are cases where the response was "Closed with explanation" but the delays or unresolved issues strongly imply they were not handled promptly or adequately.\n\nOverall, the evidence suggests that at least some complaints were not handled in a timely manner, as indicated directly by the res

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, often related to mismanagement, misinformation, and systemic issues. Based on the provided complaints, common reasons include:\n\n1. **Lack of Notification and Communication:** Many borrowers were not properly notified about loan transfers, due dates, or repayment start dates, leading to unintentional delinquency and missed payments.\n\n2. **Misleading or Incomplete Information:** Borrowers reported receiving incorrect or misleading information about their loan balances, repayment obligations, or eligibility for programs like income-driven repayment or forgiveness, which caused confusion and unintended default.\n\n3. **System Errors and Technical Difficulties:** Issues such as online portal lockouts, incorrect account statuses, and errors in reporting contributed to borrowers not making payments or being marked delinquent improperly.\n\n4. **Inadequate Support and Assistance:** Borrowers often found customer service unhelpful,

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [49]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [50]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [51]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [52]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [56]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issues with loans appear to be related to difficulties in communication and account management, such as:\n\n- Struggling to repay loans due to errors or issues with payment plans.\n- Problems with loan reporting, including incorrect or improper reporting of account status or default.\n- Difficulties in obtaining clear information about loan balances, loan servicer changes, or payment amounts.\n- Issues with loan servicing companies failing to respond appropriately or failing to verify or process applications.\n- Unauthorized or illegal reporting and collection practices, including violations of privacy laws.\n\nWhile these are specific to student loans in the context provided, a recurring theme is that many complaints involve mismanagement, lack of transparency, or errors in the handling of loans and related information. \n\nTherefore, a common underlying issue with loans, especially highlighted here, is **mismanagement or errors in se

In [57]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that many complaints were responded to in a timely manner, with responses marked as \'Yes\' under the \'Timely response?\' field. Notably, several complaints state "Closed with explanation," indicating that they were addressed within the required time frame. \n\nHowever, there is at least one complaint regarding a lack of response or handling—specifically, the complaint about Nelnet (row 17). The consumer\'s narrative details multiple issues with lack of responses and conduct that suggests their complaint was not handled promptly or satisfactorily.\n\nIn summary:\n\n- Multiple complaints confirm responses were handled in a timely manner.\n- One complaint (about Nelnet\'s failure to respond to Certified Mail and ongoing misconduct) indicates that the complaint was not properly handled or responded to, suggesting that some complaints did not get handled in a timely manner.\n\nTherefore, yes, some complaints did not get handled in a timely man

In [58]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues such as difficulties dealing with their loan servicers, miscommunications or inadequate information about their loan status, problems with payment processing, and disputes over the legitimacy or accuracy of their loan details. Some specific reasons noted in the complaints include receiving bad information about loan statuses, delays or errors in re-amortizing payments after forbearance ended, and inaccurate reports of default or delinquency. Additionally, instances of alleged mismanagement, lack of transparency, or improper handling of personal data have also contributed to borrowers' difficulties in repayment."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer:

In this instance, semantic chunking might produce a lot of small chunks that are similar in meaning. In term, this might fail to capture a broader context because each chunk is almost identical in meaning.

A solution to this could be to widen the chunking size by every 3-4 sentences (instead of every sentence), so that the model can gain more relevant context. Furthermore, you could implement a system where the model will combine chunks that have similar cosine similarity scores, reducing noise and repetition.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [129]:
import os
import getpass

In [130]:
os.environ["LANGSMITH_PROJECT"] = "Advanced RAG "
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [86]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [131]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=10)

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 99b22209-7578-4a2d-b394-d78a4fa16b84 does not have a summary. Skipping filtering.
Node 9f411f5e-f1ae-4fa6-8fba-ea6ede152bca does not have a summary. Skipping filtering.
Node 0c2fd376-c58e-43e6-8533-f294b33d12aa does not have a summary. Skipping filtering.
Node 4fb5fb4b-634f-478c-8188-a97b11bc07cd does not have a summary. Skipping filtering.
Node fe6bd02b-6cd0-4ca7-aaf3-5d9b84d9a0e5 does not have a summary. Skipping filtering.
Node b39c102f-8ee6-475b-9233-c541579e9e3e does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/51 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [132]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What are the recent changes to the federal stu...,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,Why is Aidvantage billing borrowers like me fo...,[I submitted my annual Income-Driven Repayment...,Aidvantage has not processed the IDR applicati...,single_hop_specifc_query_synthesizer
2,How does FERPA protect my personal and financi...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,How does 15 U.S.C. 16811 relate to the reinves...,[I am writing to formally dispute inaccurate i...,"15 U.S.C. 16811, part of the Fair Credit Repor...",single_hop_specifc_query_synthesizer
4,"How do the challenges faced by borrowers, such...",[<1-hop>\n\nI have provided documentation rela...,"The challenges faced by borrowers, including r...",multi_hop_abstract_query_synthesizer
5,How do the rejections of documents and issues ...,[<1-hop>\n\nI have provided documentation rela...,"The rejections of documents, such as the incom...",multi_hop_abstract_query_synthesizer
6,How did the errors and misconduct by XXXX impa...,[<1-hop>\n\nThe federal student loan COVID-19 ...,"The errors on the account, including acts of d...",multi_hop_abstract_query_synthesizer
7,H0w do payment processing failures by NelNet r...,[<1-hop>\n\nI keep setting up auto-debit with ...,"Payment processing failures by NelNet, such as...",multi_hop_abstract_query_synthesizer
8,How do the violations of 15 U.S.C. 1681i and 1...,[<1-hop>\n\nIllegal Student Loan Reporting & C...,"The violations of 15 U.S.C. 1681i, which manda...",multi_hop_specific_query_synthesizer
9,How does the transfer of the student loan acco...,[<1-hop>\n\nXX/XX/XXXX I increased the amount ...,The account was transferred to Nelnet from XXX...,multi_hop_specific_query_synthesizer


In [133]:
all_retrievers = {
    "Naive": naive_retrieval_chain,
    "Bm25": bm25_retrieval_chain,
    "Contextual Compression": contextual_compression_retrieval_chain,
    "Multi Query": multi_query_retrieval_chain,
    "Parent Document": parent_document_retrieval_chain,
    "Ensemble": ensemble_retrieval_chain,
}

In [136]:
import os
import copy
from langchain.callbacks.tracers import LangChainTracer
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    LLMContextRecall, FactualCorrectness, 
    ResponseRelevancy, ContextEntityRecall
)
from ragas import evaluate, RunConfig, EvaluationDataset

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", max_tokens=8192,
    temperature=0,
    request_timeout=600,
    verbose=True))
custom_run_config = RunConfig(timeout=600)

evaluation_results = {}

for name, retriever in all_retrievers.items():
    print(f"Evaluating {name}")
    dataset_copy = copy.deepcopy(dataset)

    for test_row in list(dataset_copy):
        response = retriever.invoke({"question": test_row.eval_sample.user_input})
        test_row.eval_sample.response = response["response"].content
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in response["context"]]

    project_name = os.environ.get("LANGSMITH_PROJECT", "Advanced RAG")
    new_trace = LangChainTracer(project_name=f"{project_name} - {name}")
    ragas_samples = [sample.eval_sample for sample in dataset_copy]
    evaluation_dataset = EvaluationDataset(ragas_samples)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[
            LLMContextRecall(),
            FactualCorrectness(),
            ResponseRelevancy(),
            ContextEntityRecall(),
        ],
        llm=evaluator_llm,
        run_config=custom_run_config,
        callbacks=[new_trace]
    )

    evaluation_results[name] = result

print(evaluation_results)

Evaluating Naive


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Evaluating Bm25


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Evaluating Contextual Compression


...

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Evaluating Ensemble


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[7]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[15]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


{'Naive': {'context_recall': 0.8889, 'factual_correctness(mode=f1)': 0.5483, 'answer_relevancy': 0.8772, 'context_entity_recall': 0.4435}, 'Bm25': {'context_recall': 0.6806, 'factual_correctness(mode=f1)': 0.4992, 'answer_relevancy': 0.8114, 'context_entity_recall': 0.4138}, 'Contextual Compression': {'context_recall': 0.7847, 'factual_correctness(mode=f1)': 0.5783, 'answer_relevancy': 0.7858, 'context_entity_recall': 0.5134}, 'Multi Query': {'context_recall': 0.8889, 'factual_correctness(mode=f1)': 0.5375, 'answer_relevancy': 0.9537, 'context_entity_recall': 0.5236}, 'Parent Document': {'context_recall': 0.7917, 'factual_correctness(mode=f1)': 0.4617, 'answer_relevancy': 0.9519, 'context_entity_recall': 0.4045}, 'Ensemble': {'context_recall': 0.8889, 'factual_correctness(mode=f1)': 0.4900, 'answer_relevancy': 0.8770, 'context_entity_recall': 0.4814}}


In [ ]:
import pandas as pd

data = {
    'Naive': {
        'context_recall': 0.8889,
        'factual_correctness': 0.5483,
        'answer_relevancy': 0.8772,
        'context_entity_recall': 0.4435
    },
    'Bm25': {
        'context_recall': 0.6806,
        'factual_correctness': 0.4992,
        'answer_relevancy': 0.8114,
        'context_entity_recall': 0.4138
    },
    'Contextual Compression': {
        'context_recall': 0.7847,
        'factual_correctness': 0.5783,
        'answer_relevancy': 0.7858,
        'context_entity_recall': 0.5134
    },
    'Multi Query': {
        'context_recall': 0.8889,
        'factual_correctness': 0.5375,
        'answer_relevancy': 0.9537,
        'context_entity_recall': 0.5236
    },
    'Parent Document': {
        'context_recall': 0.7917,
        'factual_correctness': 0.4617,
        'answer_relevancy': 0.9519,
        'context_entity_recall': 0.4045
    },
    'Ensemble': {
        'context_recall': 0.8889,
        'factual_correctness': 0.4900,
        'answer_relevancy': 0.8770,
        'context_entity_recall': 0.4814
    }
}

# Convert and flatten the DataFrame
df = pd.DataFrame(data).T.round(4)
flat_df = df.stack().to_frame().T
flat_df.columns = [f'{index[0]} - {index[1]}' for index in flat_df.columns]

# Create header and row strings with borders
headers = list(flat_df.columns)
values = flat_df.iloc[0].tolist()

# Print with borders
border = '+'.join(['-' * (len(h) + 2) for h in headers])
header_row = '|' + '|'.join([f' {h} ' for h in headers]) + '|'
value_row = '|' + '|'.join([f' {v:.4f} '.ljust(len(h)+2) for v, h in zip(values, headers)]) + '|'

print('+' + border + '+')
print(header_row)
print('+' + border + '+')
print(value_row)
print('+' + border + '+')


                        context_recall  factual_correctness  answer_relevancy  \
Retriever                                                                       
Naive                           0.8889               0.5483            0.8772   
Bm25                            0.6806               0.4992            0.8114   
Contextual Compression          0.7847               0.5783            0.7858   
Multi Query                     0.8889               0.5375            0.9537   
Parent Document                 0.7917               0.4617            0.9519   
Ensemble                        0.8889               0.4900            0.8770   

                        context_entity_recall  
Retriever                                      
Naive                                  0.4435  
Bm25                                   0.4138  
Contextual Compression                 0.5134  
Multi Query                            0.5236  
Parent Document                        0.4045  
Ensemble       